In [ ]:
# Databricks notebook sourceMAGIC %md# 🟩 Cell 1 — Imports & Config

In [ ]:
MAGIC %run /Workspace/Repos/karthikvanapalli1505@gmail.com/Bupa_Insuarnce_Project/02_Silver_Layer/_00_utils_silver

In [ ]:
# --- Imports & shared utils ---from pyspark.sql import functions as F, Window# --- Databases & paths ---DB_SILVER = "bupa_silver"DB_REF    = "bupa_reference"# ADLS external location for claims silver (adjust if needed)PATH_CLAIMS = "abfss://silver@bupaprocesseddatastorage.dfs.core.windows.net/claims"print("✅ Utils imported; DB_SILVER =", DB_SILVER, "| DB_REF =", DB_REF)

%md# Cell 2 — Load base data (Silver + Reference)

In [ ]:
claims    = spark.table(f"{DB_SILVER}.claims")providers = spark.table(f"{DB_SILVER}.providers")mapping   = spark.table(f"{DB_REF}.map_provider_id")  # created in your 01_* mapping setupprint({    "claims_rows": claims.count(),    "providers_rows": providers.count(),    "mapping_rows": mapping.count()})

%md# Cell 3 — Canonical helpers & canonicalize keys

In [ ]:
def canon(colname: str):    # Uppercase, trim, keep A–Z/0–9/_ only    return F.upper(F.regexp_replace(F.trim(F.col(colname)), r"[^A-Z0-9_]", ""))# Canonicalized views for joiningclaims_c = claims.withColumn("Provider_ID_canon", canon("Provider_ID"))prov_c   = (providers            .withColumn("Provider_ID_canon", canon("Provider_ID"))            .select("Provider_ID","Provider_ID_canon")            .dropDuplicates())mp_c     = (mapping            .withColumn("source_canon", canon("source_provider_id"))            .withColumn("target_canon", canon("target_provider_id"))            .select("source_canon","target_canon"))            print("✅ Canonicalized frames ready.")

%md# Cell 4 — Resolve claims → providers via mapping (build claims_resolved)

In [ ]:
# === Cell 4 — Resolve claims → providers via mapping (robust) ===from pyspark.sql import functions as F# 0) Safety: capture original claims columns once_claim_cols = claims.columnsif "Provider_ID" not in _claim_cols:    raise Exception("Expected column 'Provider_ID' not found in bupa_silver.claims")# 1) Build resolved join graph (claims_c -> mapping -> providers)#    If your mapping table is empty, we still proceed (left joins).resolved = (    claims_c.alias("c")    .join(mp_c.alias("m"),          F.col("c.Provider_ID_canon") == F.col("m.source_canon"),          "left")    .join(        prov_c.alias("p"),        F.col("m.target_canon") == F.col("p.Provider_ID_canon"),        "left"    ))# 2) Persist BOTH: raw + master; standardize Provider_ID = master for downstream joins#    (keep Provider_ID_raw for lineage; keep Provider_ID_master for explicit audits)#    IMPORTANT: Always reference with alias ("c." and "p.") to avoid ambiguity.claims_resolved = (    resolved      # carry all original claim columns EXCEPT the old Provider_ID (we’ll rebuild it)      .select(*[F.col(f"c.{c}").alias(c) for c in _claim_cols if c != "Provider_ID"],              # add raw and master              F.col("c.Provider_ID").alias("Provider_ID_raw"),              F.col("p.Provider_ID").alias("Provider_ID_master"))      # standardize the FK to the master value      .withColumn("Provider_ID", F.col("Provider_ID_master"))      # final column order: original (minus old Provider_ID) + standardized + lineage columns      .select(          *[F.col(c) for c in _claim_cols if c != "Provider_ID"],          "Provider_ID",            # mastered FK for all downstream joins          "Provider_ID_raw",        # lineage (as in claims source)          "Provider_ID_master"      # explicit master (same as Provider_ID)      ))# 3) Quick audit so you can see if mapping appliedaudit = {    "rows_out": claims_resolved.count(),    "master_null_rows": claims_resolved.filter(F.col("Provider_ID").isNull()).count(),    "raw_null_rows": claims_resolved.filter(F.col("Provider_ID_raw").isNull()).count()}print("Cell 4 audit →", audit)# 4) (Optional) Tiny sample to eyeball the conversiondisplay(claims_resolved.select("Provider_ID_raw","Provider_ID","Provider_ID_master").dropDuplicates().limit(10))

%md# Cell 5 — Diagnostics (mapping coverage & unresolved analysis)

In [ ]:
total_rows   = claims.count()null_in_src  = claims.filter(F.col("Provider_ID").isNull()).count()mapped_keys  = resolved.filter(F.col("m.target_canon").isNotNull()).count()matched_prov = resolved.filter(F.col("p.Provider_ID_canon").isNotNull()).count()unresolved_df = resolved.filter(F.col("p.Provider_ID_canon").isNull())unresolved_cnt = unresolved_df.count()print({  "total_claim_rows": total_rows,  "claims_with_null_provider_id": null_in_src,  "mapped_claim_rows_via_mapping": mapped_keys,  "matched_provider_rows": matched_prov,  "unresolved_rows_post_join": unresolved_cnt})# (Optional) show a few unresolved root causesif unresolved_cnt > 0:    print("Sample unresolved sources (unmapped):")    display(unresolved_df.filter(F.col("m.target_canon").isNull())                         .select("c.Provider_ID","c.Provider_ID_canon").distinct().limit(10))    print("Sample unresolved targets (mapped but missing in providers):")    display(unresolved_df.filter(F.col("m.target_canon").isNotNull())                         .select("m.target_canon").distinct().limit(10))

%md# Cell 6 — Quarantine unresolved (soft FK) + keep pipeline flowing

In [ ]:
# Soft enforcement: quarantine unresolved rows; do NOT drop from final outputif unresolved_cnt > 0:    quarantine(unresolved_df.select([F.col("c."+x) for x in claims.columns]),               "fk_provider_unresolved_soft", "claims")print("✅ Quarantine step completed (if any).")

%md# Cell 7 — Write Silver table + ADLS (Delta) and OPTIMIZE

In [ ]:
full_table = f"{DB_SILVER}.claims"# Write to metastore table (overwrite schema because we added columns)(claims_resolved.write    .format("delta")    .mode("overwrite")    .option("overwriteSchema", "true")    .saveAsTable(full_table))# Write to external ADLS path (same dataset)(claims_resolved.write    .format("delta")    .mode("overwrite")    .option("overwriteSchema", "true")    .save(PATH_CLAIMS))print("✅ Claims Silver written to table & ADLS.")# Storage hygiene (best-effort)try:    spark.sql(f"OPTIMIZE {full_table} ZORDER BY (Policy_ID, Date_Reported)")except Exception as e:    print("[WARN] OPTIMIZE skipped:", e)

%md# Cell 8 — Metrics & overlap verification

In [ ]:
from datetime import datetime# Metrics tablespark.sql(f"""CREATE TABLE IF NOT EXISTS {DB_SILVER}.__run_metrics (  metric STRING, value STRING, context STRING, ts TIMESTAMP) USING DELTA""")def write_metric(n,v,c):    (spark.createDataFrame([(n,str(v),c,datetime.utcnow())],      "metric STRING, value STRING, context STRING, ts TIMESTAMP")     .write.mode("append").saveAsTable(f"{DB_SILVER}.__run_metrics"))# Provider master resolution metric (now Provider_ID is mastered)total = claims_resolved.count()null_master = claims_resolved.filter("Provider_ID IS NULL").count()resolved_rate = 0.0 if total==0 else round(100.0*(total - null_master)/total, 2)write_metric("claims_total_rows", total, "claims_silver")write_metric("claims_provider_null_master_rows", null_master, "claims_silver")write_metric("claims_provider_master_resolved_pct", resolved_rate, "claims_silver")# Overlap check: should be > 0 now because we standardized to mastercommon = (spark.table(f"{DB_SILVER}.claims").select("Provider_ID").where("Provider_ID IS NOT NULL").distinct()          .join(spark.table(f"{DB_SILVER}.providers").select("Provider_ID").distinct(), "Provider_ID", "inner")          .count())print({  "claims_provider_master_resolved_pct": resolved_rate,  "common_provider_ids_claims_vs_providers": common})